In [1]:
from utils import *
import warnings
warnings.filterwarnings('ignore')

**UniProt Search**

In [ ]:
# # Run this cell only at the first time!
# entrez_ids = list([str(i) for i in gid2uid.keys()])
# with open(f'{root}/mapping/gid_text.txt', 'w') as f:
#     f.write(','.join(entrez_ids))

# # Then, search in UniProt

In [ ]:
# # Run this cell only at the first time!
# gid2protein_raw = pd.read_csv(f'{root}/mapping/gid2protein_raw.tsv', sep='\t')
# gid2protein = gid2protein_raw.drop_duplicates(subset='From', keep='first')
# gid2protein.to_csv(f'{root}/mapping/gid2protein.csv', index=False)

In [2]:
# NOTE: Run this cell!
gid2protein = pd.read_csv(f'{root}/mapping/gid2protein.csv')
protein_coding_gids = set(gid2protein['From'])

**Preprocess SLKB**

In [3]:
df = os.path.join(root, 'SLKB/SLKB_original_scores.csv')

df = pd.read_csv(df)
df = df[['cell_line_origin', 'gene_1', 'gene_2', 'SL_or_not']] #, 'SL_score'
df.replace({"SL_or_not": {'Not SL': 0, 'SL': 1}}, inplace=True)
df.columns = [CONTEXT_COLNAME, GENE1_COLNAME, GENE2_COLNAME, CLASSLABEL_COLNAME] #, REGLABEL_COLNAME

print("Raw SLKB has", len(df), "rows and", count_gene_num(df), "unique genes")

Raw SLKB has 280483 rows and 6125 unique genes


In [4]:
# Exact raw repeats may be removed here for efficiency only.
# Definitive unordered pair/context cleaning happens after gene-ID mapping below.
df = df.drop_duplicates(ignore_index=True)
print("After removing exact raw repeats:", len(df))

After removing exact raw repeats: 280280


In [5]:
print("Gene num after alignment:", len(set(protein_coding_gids).intersection(set(gid2uid.keys()))))

# df = df[df[GENE1_COLNAME].isin(gid2uid.keys()) & df[GENE2_COLNAME].isin(gid2uid.keys())]
df = df_gene2id(df)
print("After removing genes not in SynlethDB, SLKB has", len(df), "items and", count_gene_num(df), "unique genes")

df = df[df[GENE1_COLNAME].isin(protein_coding_gids) & df[GENE2_COLNAME].isin(protein_coding_gids)]
print("After removing non-coding genes, SLKB has", len(df), "items and", count_gene_num(df), "unique genes")

# Canonicalize AFTER ID mapping: aliases and reverse pairs share one identity.
# Same-label repeats -> one row; mixed labels -> discard the entire pair/context.
df = remove_rows_with_condition(df)
validate_clean_pairs(df)
print("After removing contradicting classes, SLKB has", len(df), "items and", count_gene_num(df), "unique genes")

Gene num after alignment: 9807
After removing genes not in SynlethDB, SLKB has 209376 items and 3017 unique genes
After removing non-coding genes, SLKB has 209328 items and 3009 unique genes
After removing contradicting classes, SLKB has 209211 items and 3009 unique genes


In [6]:
import numpy as np
import pandas as pd
import os
import pickle
# from .Struct2SL import load_data

structure_folder = "/data/qingyuyang/SL/Struct2SL/struct/feat_mat" #"./data/struct_features"
sequence_file = "/data/qingyuyang/SL/Struct2SL/sequence/seq_features.npz"  # "./data/sequence_features.npz"
ppi_file = "/data/qingyuyang/SL/Struct2SL/PPI/ppi_emb.npz" #"./data/ppi_features.npz"

structure_feat_genes = []
for file in os.listdir(structure_folder):
    gene_name = os.path.splitext(file)[0]
    structure_feat_genes.append(gene_name)

with open(ppi_file, 'rb') as f:
    ppi_features = pickle.load(f)

with open(sequence_file, 'rb') as f:
    sequence_features = pickle.load(f)

struct2sl_valid_genes = list(
    set(structure_feat_genes) & set(sequence_features.keys()) & set(ppi_features.keys())
)
print(len(struct2sl_valid_genes))
struct2sl_valid_genes = [gn2gid[i] for i in struct2sl_valid_genes if i in gn2gid.keys()]
print(len(struct2sl_valid_genes))

17200
9212


In [7]:
df = df[df[GENE1_COLNAME].isin(struct2sl_valid_genes) & df[GENE2_COLNAME].isin(struct2sl_valid_genes)]
print("After removing genes not in STRING or AFDB, SLKB has", len(df), "items and", count_gene_num(df), "unique genes")
# NOTE: This would also remove genes that only pairs with invalid genes

After removing genes not in STRING or AFDB, SLKB has 206771 items and 2967 unique genes


In [8]:
df.to_csv(f'{root}/SLKB/SLKB_processed_new.csv', index=False)

In [2]:
df = pd.read_csv(f'{root}/SLKB/SLKB_processed_new.csv')
# Reject old unclean caches if this notebook is resumed after preprocessing.
validate_clean_pairs(df)

**Count Statistics**

In [2]:
root_path = f'{root}/SLKB'

table_save_path = f'{root_path}/cell_line/table'
os.makedirs(table_save_path, exist_ok=True)

fig_save_path = f'{root_path}/cell_line/gene_count'
os.makedirs(fig_save_path, exist_ok=True)

In [4]:
cell_lines = list(set(df[CONTEXT_COLNAME]))
print('In total', len(cell_lines), 'cell lines')

In total 22 cell lines


In [5]:
def visualize_gene_freq(gene_count: dict, max_gene_show: int = 200, show_yval_step: int = 10, show_xlabel_step: int = 5, 
                        plot: bool = False, save_path: Path | None = None, title: str | None = None):
    gene_num = len(gene_count)
    jump = int(gene_num / max_gene_show) if gene_num >= max_gene_show else 1
    sorted_list = sorted(gene_count.items(), key=lambda item: item[1], reverse=True)[::jump]
    keys, values = zip(*sorted_list)
    keys = [gid2gn[i] for i in keys]

    fig, ax = plt.subplots(figsize=(20, 8), dpi=350)
    bars = ax.bar(keys, values)
    for i, bar in enumerate(bars):
        if i % show_yval_step == 0:
            yval = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, yval, int(yval), ha='center', va='bottom')
    
    # if title:
    #     plt.title(f"{title}")
    plt.xlabel('Genes')
    plt.ylabel('Number of appearances in SL data')
    plt.xticks(np.arange(len(keys))[::show_xlabel_step], keys[::show_xlabel_step], rotation=90)
    
    if save_path:
        plt.savefig(f"{save_path}/{title}.jpg")
    if not plot:
        plt.clf()
        plt.close(fig)
    else:
        plt.show()

In [6]:
omics_root = '/data/qingyuyang/SL/MVGCNiSL/data/cellline_feats'
have_omics_cl = set([fn.split('_')[0] for fn in os.listdir(omics_root)])

In [7]:
column_names = ['cell_line', '# pairs', '# pos', '# neg', 'n/p ratio', '# genes', '# unique genes', 'have_omics']
stat_df = pd.DataFrame(0, index=range(len(cell_lines)), columns=column_names)

for i in tqdm(range(len(cell_lines))):
    cl = cell_lines[i]
    # print(cl)
    path = f'{table_save_path}/{cl}.csv'

    cl_df = select_cl_from_slkb(df, cl)
    # cl_df = remove_rows_with_condition(cl_df)
    cl_df.to_csv(path, index=False)

    total, pos, neg, ratio = count_pn_ratio(cl_df)
    gene_list, gene_set, gene_count = count_gene_freq(cl_df)
    unique_gene_num = list(gene_count.values()).count(1)

    if_cell_embed = 1 if cl in have_omics_cl else 0
    stat_df.iloc[i] = [cl, total, pos, neg, ratio, len(gene_set), unique_gene_num, if_cell_embed]

    visualize_gene_freq(gene_count, save_path=f'{fig_save_path}', title=cl)

stat_df.sort_values(by=["# unique genes", "# genes"], inplace=True, ascending=False, ignore_index=True)
stat_df.to_csv(f'{table_save_path}/overall_statistics.csv', index=False)
stat_df

  0%|          | 0/22 [00:00<?, ?it/s]

100%|██████████| 22/22 [00:23<00:00,  1.09s/it]


,cell_line,# pairs,# pos,# neg,n/p ratio,# genes,# unique genes,have_omics
0,MEWO,2443,270,2173,8.0481,2088,1177,1
1,PATU8988S,1915,329,1586,4.8207,1565,775,1
2,HS936T,1915,332,1583,4.7681,1565,775,1
3,MEL202,1915,370,1545,4.1757,1565,775,1
4,IPC298,1915,312,1603,5.1378,1565,775,1
5,HS944T,1915,275,1640,5.9636,1565,775,1
6,HSC5,1915,365,1550,4.2466,1565,775,1
7,PK1,1915,361,1554,4.3047,1565,775,1
8,GI1,1915,235,1680,7.1489,1565,775,1
9,MELJUSO,2189,276,1913,6.9312,1571,772,1


From the above table, we found that:
1. HT29 and 786O are unsuitable for SL prediction because of few positive samples;
2. Cell-line 15-19 has no 'tail' scenes;
3. Genes in cell-line 1-8 are the same, suitable for transfer scenes;
4. PC9's genes are all unique, so its C1 and Tail are equivalent and doesn\'t have C2.

**Cut C1/C2/Tail scenes**

In [3]:
stat_df = pd.read_csv(f'{table_save_path}/overall_statistics.csv')
new_cell_lines = stat_df['cell_line'][:-2]

In [5]:
specific_save_path = f'{root_path}/specific'
os.makedirs(specific_save_path, exist_ok=True)

for cl in tqdm(new_cell_lines):
    cl_df = pd.read_csv(f'{table_save_path}/{cl}.csv')

    split_instance = SpecificSplit(cl_df, f'{specific_save_path}/{cl}')
    split_instance.all_scenes()

 80%|████████  | 16/20 [01:03<00:42, 10.64s/it]

This cell line is unsuitable for long-tail scene!


 85%|████████▌ | 17/20 [01:19<00:37, 12.34s/it]

This cell line is unsuitable for long-tail scene!


 90%|█████████ | 18/20 [01:20<00:18,  9.02s/it]

This cell line is unsuitable for long-tail scene!


 95%|█████████▌| 19/20 [01:21<00:06,  6.49s/it]

This cell line is unsuitable for long-tail scene!


100%|██████████| 20/20 [01:21<00:00,  4.10s/it]

This cell line is unsuitable for long-tail scene!


In [6]:
count_specific_statistics(specific_save_path)

80it [00:26,  3.03it/s]


,cell_line,scene,fold,set,# pairs,# pos,# neg,n/p ratio,# genes,# unique genes
0,MEWO,C1,0,0,1563,173,1390,8.0347,1590,977
1,MEWO,C1,0,1,391,43,348,8.0930,593,472
2,MEWO,C1,0,2,489,54,435,8.0556,753,611
3,MEWO,C1,1,0,1563,173,1390,8.0347,1604,978
4,MEWO,C1,1,1,391,43,348,8.0930,592,459
...,...,...,...,...,...,...,...,...,...,...
1120,22RV1,C3,3,1,21,8,13,1.6250,7,0
1121,22RV1,C3,3,2,36,17,19,1.1176,9,0
1122,22RV1,C3,4,0,378,122,256,2.0984,28,0
1123,22RV1,C3,4,1,28,3,25,8.3333,8,0


**Form OnetoOne scenes**

In [4]:
import itertools

transfer_save_path = f'{root_path}/onetoone'
os.makedirs(transfer_save_path, exist_ok=True)

# cell_names = ['A549', 'GI1', 'MEL202']
combinations = list(itertools.combinations(new_cell_lines, 2))
new_comb = [(i[1], i[0]) for i in combinations]
whole_comb = combinations + new_comb

for source, target in tqdm(whole_comb):
    output_dir = f'{transfer_save_path}/{source}_{target}'
    os.makedirs(output_dir, exist_ok=True)

    source_df = pd.read_csv(f'{table_save_path}/{source}.csv')
    target_df = pd.read_csv(f'{table_save_path}/{target}.csv')
    split_instance = OnetoOneSplit(source_df, target_df, output_dir)
    split_instance.split()

100%|██████████| 380/380 [00:44<00:00,  8.55it/s]


**Form MultitoOne scenes**

In [ ]:
new_cell_lines = ['A375', 'A549', 'JURKAT', 'MEWO', '22RV1', 'RK1']

multitoone_save_path = f'{root_path}/multitoone'
os.makedirs(multitoone_save_path, exist_ok=True)

multi_cell_lines = list(new_cell_lines)
for target in tqdm(multi_cell_lines):
    source_contexts = [context for context in multi_cell_lines if context != target]
    source_dfs = {
        context: pd.read_csv(f'{table_save_path}/{context}.csv')
        for context in source_contexts
    }
    target_df = pd.read_csv(f'{table_save_path}/{target}.csv')

    output_dir = f'{multitoone_save_path}/Multi_{len(source_dfs)}_to_{target}'
    split_instance = MultitoOneSplit(
        source_dfs,
        target_df,
        target_context=target,
        save_path=output_dir,
    )
    split_instance.split()

100%|██████████| 20/20 [00:22<00:00,  1.12s/it]


**1:1**

In [7]:
balanced_table_save_path = f'{root_path}/cell_line/table_1:1'
os.makedirs(balanced_table_save_path, exist_ok=True)

# Only cells selected for the experiments; single-class cells cannot form 1:1 data.
for cl in tqdm(new_cell_lines):
    cell_line_df = pd.read_csv(f'{table_save_path}/{cl}.csv')
    balanced_df = balance_binary_pairs(cell_line_df, random_seed=43)
    balanced_df.to_csv(f'{balanced_table_save_path}/{cl}.csv', index=False)

100%|██████████| 20/20 [00:00<00:00, 65.17it/s]


In [5]:
table_save_path_1 = f'{root_path}/cell_line/table_1:1'

In [9]:
specific_save_path_1 = f'{root_path}/specific_1:1'
os.makedirs(specific_save_path_1, exist_ok=True)

for cl in tqdm(new_cell_lines):
    cl_df = pd.read_csv(f'{table_save_path_1}/{cl}.csv')

    split_instance = SpecificSplit(cl_df, f'{specific_save_path_1}/{cl}')
    split_instance.all_scenes()

 80%|████████  | 16/20 [00:13<00:03,  1.07it/s]

This cell line is unsuitable for long-tail scene!


 90%|█████████ | 18/20 [00:13<00:01,  1.77it/s]

This cell line is unsuitable for long-tail scene!
This cell line is unsuitable for long-tail scene!


100%|██████████| 20/20 [00:13<00:00,  1.43it/s]

This cell line is unsuitable for long-tail scene!
This cell line is unsuitable for long-tail scene!


In [6]:
import itertools

transfer_save_path_1 = f'{root_path}/onetoone_1:1'
os.makedirs(transfer_save_path_1, exist_ok=True)

# cell_names = ['A549', 'GI1', 'MEL202']
combinations = list(itertools.combinations(new_cell_lines, 2))
new_comb = [(i[1], i[0]) for i in combinations]
whole_comb = combinations + new_comb

for source, target in tqdm(whole_comb):
    output_dir = f'{transfer_save_path_1}/{source}_{target}'
    os.makedirs(output_dir, exist_ok=True)

    source_df = pd.read_csv(f'{table_save_path_1}/{source}.csv')
    target_df = pd.read_csv(f'{table_save_path_1}/{target}.csv')
    split_instance = OnetoOneSplit(source_df, target_df, output_dir)
    split_instance.split()

100%|██████████| 380/380 [00:09<00:00, 40.08it/s]


In [13]:
count_specific_statistics(specific_save_path_1)

80it [00:04, 18.13it/s]


,cell_line,scene,fold,set,# pairs,# pos,# neg,n/p ratio,# genes,# unique genes
0,MEWO,C1,0,0,355,177,178,1.0056,456,332
1,MEWO,C1,0,1,89,45,44,0.9778,153,134
2,MEWO,C1,0,2,112,56,56,1.0000,188,159
3,MEWO,C1,1,0,356,177,179,1.0113,479,369
4,MEWO,C1,1,1,89,45,44,0.9778,154,133
...,...,...,...,...,...,...,...,...,...,...
1120,22RV1,C3,3,1,12,8,4,0.5000,6,0
1121,22RV1,C3,3,2,23,17,6,0.3529,9,0
1122,22RV1,C3,4,0,201,122,79,0.6475,28,0
1123,22RV1,C3,4,1,12,3,9,3.0000,8,1


In [ ]:
multitoone_save_path_1 = f'{root_path}/multitoone_1:1'
os.makedirs(multitoone_save_path_1, exist_ok=True)

# Explicit panel avoids inheriting a stale cell list from earlier scenes.
multi_cell_lines = ['A375', 'A549', 'JURKAT', 'MEWO', '22RV1', 'PK1']
table_save_path_1 = f'{root_path}/cell_line/table_1:1'
for target in tqdm(multi_cell_lines):
    source_contexts = [context for context in multi_cell_lines if context != target]
    source_dfs = {
        context: pd.read_csv(f'{table_save_path_1}/{context}.csv')
        for context in source_contexts
    }
    target_df = pd.read_csv(f'{table_save_path_1}/{target}.csv')

    output_dir = f'{multitoone_save_path_1}/Multi_{len(source_dfs)}_to_{target}'
    split_instance = MultitoOneSplit(
        source_dfs,
        target_df,
        target_context=target,
        save_path=output_dir,
        balanced=True,  # Exact 1:1 in every train/val/test split and source context.
    )
    split_instance.split()